# Laboratorio: Búsqueda Local - Problema de las 8 Reinas
- Adrián Ricardo González Muralles - 23152
- Jose Pablo Ordoñez Barrios - 231329

Este notebook implementa y compara algoritmos de búsqueda local:
- Hill Climbing
- Hill Climbing con Random Restart
- Beam Search (k = 2, 5, 10)

In [26]:
import random
import time
import numpy as np 

## Funciones base

In [27]:
def es_solucion(estado):
    n = len(estado)
    for c1 in range(n):
        for c2 in range(c1 + 1, n):
            if estado[c1] == estado[c2] or abs(estado[c1] - estado[c2]) == abs(c1 - c2):
                return False
    return True

def heuristica(estado):
    conflictos = 0
    n = len(estado)
    for c1 in range(n):
        for c2 in range(c1 + 1, n):
            if estado[c1] == estado[c2] or abs(estado[c1] - estado[c2]) == abs(c1 - c2):
                conflictos += 1
    return conflictos

def vecinos(estado):
    vecinos = []
    n = len(estado)
    for col in range(n):
        for fila in range(n):
            if fila != estado[col]:
                nuevo = estado.copy()
                nuevo[col] = fila
                vecinos.append(nuevo)
    return vecinos

## Hill Climbing

In [28]:
def hill_climbing(estado):
    actual = estado
    pasos = 0
    while True:
        vecinos_list = vecinos(actual)
        mejor = min(vecinos_list, key=heuristica)
        if heuristica(mejor) >= heuristica(actual):
            break
        actual = mejor
        pasos += 1
        if heuristica(actual) == 0:
            break
    return actual, pasos

## Random Restart

In [29]:
def random_restart(runs=10):
    for _ in range(runs):
        estado = [random.randint(0,7) for _ in range(8)]
        sol, pasos = hill_climbing(estado)
        if heuristica(sol) == 0:
            return sol, pasos
    return sol, pasos

## Beam Search

In [30]:
def beam_search(estado, k=3, max_iter=100):
    beam = [[random.randint(0,7) for _ in range(8)] for _ in range(k)]
    pasos = 0

    for _ in range(max_iter):
        for b in beam:
            if heuristica(b) == 0:
                return b, pasos
        
        todos_vecinos = []
        for b in beam:
            todos_vecinos.extend(vecinos(b))
        
        todos_vecinos.sort(key=heuristica)
        
        nuevo_beam = todos_vecinos[:k]
        
        if min(map(heuristica, nuevo_beam)) >= min(map(heuristica, beam)):
            break
        
        beam = nuevo_beam
        pasos += 1

    mejor = min(beam, key=heuristica)
    return mejor, pasos

## Experimentos

In [31]:
def experimento(func, runs=1000):
    exitos = 0
    pasos_total = 0
    h_total = 0
    tiempo_total = 0
    
    for _ in range(runs):
        estado = [random.randint(0,7) for _ in range(8)]
        inicio = time.time()
        sol, pasos = func(estado)
        fin = time.time()
        
        if heuristica(sol) == 0:
            exitos += 1
        pasos_total += pasos
        h_total += heuristica(sol)
        tiempo_total += (fin - inicio)
    
    return {
        'exito': exitos / runs,
        'pasos': pasos_total / runs,
        'heuristica': h_total / runs,
        'tiempo': tiempo_total / runs
    }

## Resultados

In [25]:
print('Hill Climbing:', experimento(hill_climbing))
print('Random Restart:', experimento(lambda s: random_restart()))
print('Beam Search:', experimento(lambda s: beam_search(s, k=10)))

Hill Climbing: {'exito': 0.165, 'pasos': 3.252, 'heuristica': 1.233, 'tiempo': 0.0004747326374053955}
Random Restart: {'exito': 0.8, 'pasos': 3.865, 'heuristica': 0.283, 'tiempo': 0.002471621513366699}
Beam Search: {'exito': 0.495, 'pasos': 2.522, 'heuristica': 0.555, 'tiempo': 0.003443791151046753}


donde:
- exito = porcentaje de veces que encontró solución.
- pasos = cantidad promedio de movimientos que hizo el algoritmo.
- heuristica = número promedio de conflictos al final en la que 0 sería una solución perfecta.
- tiempo = tiempo promedio por cada ejecución (en segundos).

## Discusión
#### ¿Qué tan frecuentemente Hill Climbing encuentra una solución?

Hill Climbing encuentra soluciones con baja frecuencia, alrededor del 13.7%, ya que suele quedarse estancado antes de llegar a un estado óptimo.

#### ¿Qué tipo de problemas presenta Hill Climbing en este dominio?

Presenta problemas como óptimos locales, mesetas y crestas, lo que impide que el algoritmo continúe mejorando y alcance la solución.

#### ¿La variante elegida mejora el desempeño? ¿Por qué?

Sí, Random Restart mejora el desempeño porque reinicia la búsqueda varias veces, aumentando la probabilidad de encontrar una solución y evitando quedar atrapado.

#### ¿Cómo afecta el valor de k en Beam Search?

Un k mayor permite explorar más estados y aumenta la probabilidad de éxito, pero también incrementa el costo computacional.

#### ¿Cuál algoritmo resultó más efectivo?

Random Restart fue el más efectivo, ya que obtuvo la mayor tasa de éxito entre los algoritmos evaluados.

#### ¿Qué relación existe entre costo computacional y tasa de éxito?

A mayor exploración del espacio de búsqueda, mayor tasa de éxito, pero también mayor costo computacional.